# Einx vs Einops 关系说明

参考: https://einx.readthedocs.io/en/stable/gettingstarted/tutorial_notation.html

## 核心关系

**Einx 是基于 Einops 的扩展库**，它们的关系是：

- ✅ **Einops**: 基础库，提供 `rearrange`, `reduce`, `repeat` 等操作
- ✅ **Einx**: 在 Einops 基础上扩展，增加了更多功能和更强大的语法
- ✅ **兼容性**: Einx 完全兼容 Einops 的语法，可以无缝替换使用


In [ ]:
# 导入两个库
!pip install einops einx

import torch
from einops import rearrange, reduce, repeat  # Einops
import einx  # Einx (扩展版)


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://mirrors.ustc.edu.cn/pypi/simple
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [einx]
✅ 两个库都可以使用！


## 1. 基本兼容性 - Einx 完全支持 Einops 语法

Einx 可以完全替代 Einops，所有 Einops 代码都能在 Einx 中运行：


In [3]:
x = torch.randn(2, 3, 4, 5)

# Einops 写法
y1 = rearrange(x, 'b c h w -> b h w c')
print(f"Einops: {y1.shape}")

# Einx 写法 (完全兼容!)
y2 = einx.rearrange("b c h w -> b h w c", x)
print(f"Einx:   {y2.shape}")

# 结果完全一样
assert torch.allclose(y1, y2)
print("✅ 完全兼容！")


Einops: torch.Size([2, 4, 5, 3])
Einx:   torch.Size([2, 4, 5, 3])
✅ 完全兼容！


## 2. Einx 的新特性

Einx 在 Einops 基础上增加了以下新功能：

### 2.1 方括号 `[]` 表示操作轴

Einx 引入了 `[]` 语法来明确指定操作应用的轴，类似于 NumPy 的 `axis` 参数：


In [5]:
x = torch.randn(2, 3, 4)

# Einops: 需要手动指定 axis 或用 reduce
y1 = reduce(x, 'b c h -> b c', 'sum')  # 对 h 维度求和
print(f"Einops sum: {y1.shape}")  # [2, 3]

# Einx: 直接在表达式中指定 sum 的轴
y2 = einx.sum("b c [h]", x)  # [h] 表示对 h 维度操作
print(f"Einx sum:   {y2.shape}")  # [2, 3]

# 验证结果
assert torch.allclose(y1, y2)
print("✅ Einx 更简洁！")


Einops sum: torch.Size([2, 3])
Einx sum:   torch.Size([2, 3])
✅ Einx 更简洁！


In [7]:
# Einx 的 keepdims 行为更灵活
x = torch.randn(2, 3)  # 2D 张量

# 不保留维度
y1 = einx.sum("b [c]", x)
print(f"不保留维度: {y1.shape}")  # [2]

# 保留维度 (用括号包裹)
y2 = einx.sum("b ([c])", x)
print(f"保留维度:   {y2.shape}")  # [2, 1]


不保留维度: torch.Size([2])
保留维度:   torch.Size([2, 1])


### 2.2 连接操作 `+` 

Einx 支持在表达式中直接表示 concatenate 操作：


In [26]:
x = torch.randn(256, 256, 3)  # RGB 图像
y = torch.randn(256, 256)     # 灰度图

# Einops: 需要手动处理维度再 concat
import torch.nn.functional as F
y_expanded = y.unsqueeze(-1)  # [256, 256, 1]
z1 = torch.cat([x, y_expanded], dim=-1)
print(f"Einops concat: {z1.shape}")

# Einx: 直接在表达式中 concat
z2 = einx.rearrange("h w c, h w -> h w (c + 1)", x, y)
print(f"Einx concat:   {z2.shape}")

assert torch.allclose(z1, z2)
print("✅ Einx 更直观！")


Einops concat: torch.Size([256, 256, 4])
Einx concat:   torch.Size([256, 256, 4])
✅ Einx 更直观！


### 2.3 更多内置操作

Einx 提供了更多内置的 tensor 操作，如 `einx.sum`, `einx.mean`, `einx.softmax`, `einx.flip` 等，可以直接在表达式中使用：


In [27]:
# Attention 中的 softmax
attn_logits = torch.randn(2, 8, 10, 10)  # [batch, heads, queries, keys]

# Einops: 需要手动处理
attn1 = rearrange(attn_logits, 'b h q k -> b h q k')
attn1 = attn1.softmax(dim=-1)

# Einx: 直接在表达式中
attn2 = einx.softmax("b h q [k]", attn_logits)

assert torch.allclose(attn1, attn2)
print("✅ Einx 更简洁！")


✅ Einx 更简洁！


## 3. CS336 作业文档中的例子

以下是作业文档中给出的 einops/einx 实际应用示例：

### 3.1 批量矩阵乘法 (Batched Matrix Multiplication)


In [ ]:
from einops import einsum

# 场景: 线性层的前向传播
batch, sequence, d_in, d_out = 4, 10, 64, 128
D = torch.randn(batch, sequence, d_in)  # 输入
A = torch.randn(d_out, d_in)            # 权重矩阵

# ❌ 传统写法 - 难以理解输入输出形状
Y_basic = D @ A.T
print(f"传统写法: {Y_basic.shape}")

# ✅ Einsum 写法 - 自文档化，清晰明了
Y_einsum = einsum(D, A, "batch sequence d_in, d_out d_in -> batch sequence d_out")
print(f"Einsum:   {Y_einsum.shape}")

# ✅ 支持任意前导维度的版本
Y_flex = einsum(D, A, "... d_in, d_out d_in -> ... d_out")
print(f"灵活版:   {Y_flex.shape}")

assert torch.allclose(Y_basic, Y_einsum)
print("✅ 结果一致！")


传统写法: torch.Size([4, 10, 128])
Einsum:   torch.Size([4, 10, 128])
灵活版:   torch.Size([4, 10, 128])
✅ 结果一致！


### 3.2 广播操作 - 图像调暗

场景：对一批图像生成 10 个不同亮度的版本


In [ ]:
# 场景: 生成不同亮度的图像版本
images = torch.randn(64, 128, 128, 3)  # (batch, height, width, channel)
dim_by = torch.linspace(start=0.0, end=1.0, steps=10)  # 10 个亮度系数

# ❌ 传统方法 - 需要手动 reshape 和广播
dim_value = rearrange(dim_by, "dim_value -> 1 dim_value 1 1 1")
images_rearr = rearrange(images, "b height width channel -> b 1 height width channel")
dimmed_images_v1 = images_rearr * dim_value
print(f"传统方法: {dimmed_images_v1.shape}")

# ✅ Einsum 一步搞定 - 自动处理广播
dimmed_images_v2 = einsum(
    images, dim_by,
    "batch height width channel, dim_value -> batch dim_value height width channel"
)
print(f"Einsum:   {dimmed_images_v2.shape}")

assert torch.allclose(dimmed_images_v1, dimmed_images_v2)
print("✅ 结果一致！")


### 3.3 像素混合 (Pixel Mixing)

场景：对图像的所有像素进行线性变换，但每个通道独立处理


In [31]:
# 场景: 对图像像素进行线性变换，每个通道独立
height = width = 32
channels_last = torch.randn(64, 32, 32, 3)  # (batch, height, width, channel)
B = torch.randn(32*32, 32*32)  # 像素混合矩阵

# ❌ 传统方法 - 一堆 view 和 transpose，难以理解
channels_last_flat = channels_last.view(
    -1, channels_last.size(1) * channels_last.size(2), channels_last.size(3)
)
channels_first_flat = channels_last_flat.transpose(1, 2)
channels_first_flat_transformed = channels_first_flat @ B.T
channels_last_flat_transformed = channels_first_flat_transformed.transpose(1, 2)
channels_last_transformed_v1 = channels_last_flat_transformed.view(*channels_last.shape)
print(f"传统方法: {channels_last_transformed_v1.shape}")

# ✅ Einops 方法 - 清晰的 rearrange + einsum
channels_first = rearrange(
    channels_last,
    "batch height width channel -> batch channel (height width)"
)
channels_first_transformed = einsum(
    channels_first, B,
    "batch channel pixel_in, pixel_out pixel_in -> batch channel pixel_out"
)
channels_last_transformed_v2 = rearrange(
    channels_first_transformed,
    "batch channel (height width) -> batch height width channel",
    height=height, width=width
)
print(f"Einops:   {channels_last_transformed_v2.shape}")

assert torch.allclose(channels_last_transformed_v1, channels_last_transformed_v2, atol=1e-4)
print("✅ 结果一致！Einops 版本更易读！")


传统方法: torch.Size([64, 32, 32, 3])
Einops:   torch.Size([64, 32, 32, 3])
✅ 结果一致！Einops 版本更易读！


In [24]:
# ✅✅ Einx 终极版 - 一行搞定！
channels_last_transformed_v3 = einx.dot(
    "batch row_in col_in channel, (row_out col_out) (row_in col_in) -> batch row_out col_out channel",
    channels_last, B,
    col_in=width, col_out=width
)
print(f"Einx dot: {channels_last_transformed_v3.shape}")

# 使用 atol 容忍浮点精度误差
assert torch.allclose(channels_last_transformed_v1, channels_last_transformed_v3, atol=1e-4)
print("✅ Einx 一行代码完成复杂操作！")


Einx dot: torch.Size([64, 32, 32, 3])
✅ Einx 一行代码完成复杂操作！


## 6. Einx 高级语法特性

参考: https://einx.readthedocs.io/en/stable/gettingstarted/tutorial_notation.html

### 6.1 省略号 (Ellipsis) - 处理任意维度


In [ ]:
# Einx 的省略号可以匹配任意数量的维度
# "a b..." 会根据输入张量自动展开

# 3D 张量
x3d = torch.randn(2, 3, 4)
print(f"3D 输入: {x3d.shape}")
print(f"匹配 'a b...': {einx.matches('a b...', x3d)}")  # 展开为 "a b.0 b.1"

# 4D 张量
x4d = torch.randn(2, 3, 4, 5)
print(f"4D 输入: {x4d.shape}")
print(f"匹配 'a b...': {einx.matches('a b...', x4d)}")  # 展开为 "a b.0 b.1 b.2"


In [32]:
# 实用例子: 将图像/体积切分为 patches (同一个表达式适用于 2D 和 3D!)

# 2D 图像切分为 patches
img_2d = torch.randn(256, 256, 3)  # (H, W, C)
patches_2d = einx.rearrange("(s p)... c -> (s...) p... c", img_2d, p=8)
print(f"2D 图像 {img_2d.shape} -> patches {patches_2d.shape}")
# 256/8=32, 32*32=1024 个 8x8 patches

# 3D 体积切分为 cubes (同样的表达式!)
vol_3d = torch.randn(64, 64, 64, 3)  # (D, H, W, C)
cubes_3d = einx.rearrange("(s p)... c -> (s...) p... c", vol_3d, p=8)
print(f"3D 体积 {vol_3d.shape} -> cubes {cubes_3d.shape}")
# 64/8=8, 8*8*8=512 个 8x8x8 cubes


2D 图像 torch.Size([256, 256, 3]) -> patches torch.Size([1024, 8, 8, 3])
3D 体积 torch.Size([64, 64, 64, 3]) -> cubes torch.Size([512, 8, 8, 8, 3])


In [ ]:
# 可以直接用数字代替轴名，表示该维度的大小
x = torch.randn(2, 3, 4)

# 检查第一个维度是否为 2
print(f"匹配 '2 b c': {einx.matches('2 b c', x)}")  # True
print(f"匹配 '3 b c': {einx.matches('3 b c', x)}")  # False

# 实用: expand_dims / squeeze 的替代
x = torch.randn(2, 1, 3)
y = einx.rearrange("a 1 b -> 1 1 a b 1 5 6", x)
print(f"扩展维度: {x.shape} -> {y.shape}")


In [ ]:
# 用 + 表示沿某个轴拼接，长度是各部分之和
# (a + b) 表示 a 和 b 拼接在一起

# 拼接不同维度的张量
x = torch.randn(256, 256, 3)  # RGB
y = torch.randn(256, 256)     # 灰度
z = einx.rearrange("h w c, h w -> h w (c + 1)", x, y)
print(f"拼接: {x.shape} + {y.shape} -> {z.shape}")  # [256, 256, 4]

# 分割操作 (反向)
z = torch.randn(256, 256, 4)
x, y = einx.rearrange("h w (c + 1) -> h w c, h w", z)
print(f"分割: {z.shape} -> {x.shape}, {y.shape}")

# 广播拼接: 给所有通道追加一个常数
x = torch.randn(256, 256, 3)
const = torch.tensor([42.0])
z = einx.rearrange("... c, 1 -> ... (c + 1)", x, const)
print(f"追加常数: {x.shape} -> {z.shape}")


In [ ]:
# 当分解有歧义时，需要提供额外约束
x = torch.randn(10,)

# ❌ 失败: a*b=10，但 a 和 b 的值无法确定 (可能是 2*5, 5*2, 1*10, 10*1)
# einx.rearrange("(a b) -> a b", x)  # 会报错!

# ✅ 成功: 指定 a=5，则 b=2
y = einx.rearrange("(a b) -> a b", x, a=5)
print(f"指定 a=5: {x.shape} -> {y.shape}")

# ✅ 成功: 指定 b=2，则 a=5
y = einx.rearrange("(a b) -> a b", x, b=2)
print(f"指定 b=2: {x.shape} -> {y.shape}")

# ✅ 成功: 同时指定 (必须一致)
y = einx.rearrange("(a b) -> a b", x, a=5, b=2)
print(f"指定 a=5, b=2: {x.shape} -> {y.shape}")


### 6.5 方括号表示法 (Bracket Notation) - 指定操作轴

`[]` 用于标记操作应用的轴，相当于 NumPy 的 `axis` 参数。


In [15]:
# [] 内的轴是操作应用的轴，[] 外的轴是向量化的轴
x = torch.randn(2, 3, 4)

# einx.sum("a [b]", x) 等价于 np.sum(x, axis=1)
y1 = einx.sum("a [b] c", x)  # 对 b 轴求和
y2 = x.sum(dim=1)
print(f"sum over b: {y1.shape}")
assert torch.allclose(y1, y2)

# 对多个轴操作
y = einx.sum("a [...]", x)  # 对除了 a 之外的所有轴求和
print(f"sum over ...: {y.shape}")  # [2]


sum over b: torch.Size([2, 4])
sum over ...: torch.Size([2])


In [19]:
# 方括号可以嵌套在复杂表达式中

# 对每对元素求和 (类似 2x1 pooling)
x = torch.randn(2, 2, 16)
y = einx.sum("... (g [c])", x, c=2)
print(f"对每对求和: {x.shape} -> {y.shape}")  # [2, 2, 8]


x = torch.randn(2, 2, 16)
y = einx.sum("... ([g] c)", x, c=2)
print(f"对每对求和: {x.shape} -> {y.shape}")  # [2, 2, 2]

# Mean pooling with stride 4
x = torch.randn(4, 256, 256, 3)
y = einx.mean("b (s [ds])... c", x, ds=4)
print(f"4x4 mean pool: {x.shape} -> {y.shape}")  # [4, 64, 64, 3]


对每对求和: torch.Size([2, 2, 16]) -> torch.Size([2, 2, 8])
对每对求和: torch.Size([2, 2, 16]) -> torch.Size([2, 2, 2])
4x4 mean pool: torch.Size([4, 256, 256, 3]) -> torch.Size([4, 64, 64, 3])


In [20]:
# keepdims 行为: 用括号控制是否保留维度
x = torch.randn(16, 4)

# 不保留维度
y1 = einx.sum("b [c]", x)
print(f"不保留: {y1.shape}")  # [16]

# 保留维度 (括号包裹)
y2 = einx.sum("b ([c])", x)
print(f"保留:   {y2.shape}")  # [16, 1]

# 或者用 keepdims 参数
y3 = einx.sum("b [c]", x, keepdims=True)
print(f"keepdims: {y3.shape}")  # [16, 1]


不保留: torch.Size([16])
保留:   torch.Size([16, 1])
keepdims: torch.Size([16, 1])


### 6.6 `->` 和 `,` 的组合 (Composability)

`->` 和 `,` 可以嵌套在表达式内部，会自动展开。


In [22]:
# -> 和 , 的嵌套语法是高级特性，这里展示更实用的例子

# ===== 实用例子 1: 添加 bias (广播加法) =====
x = torch.randn(4, 10, 64)  # [batch, seq, d_model]
bias = torch.randn(64)       # [d_model]

# 使用 einx.add 添加 bias (自动广播)
# 注意: elementwise 操作不需要方括号
y = einx.add("b s c, c", x, bias)
print(f"添加 bias: {x.shape} + {bias.shape} -> {y.shape}")

# 验证: 等价于 PyTorch 的广播加法
y_torch = x + bias
assert torch.allclose(y, y_torch)
print("✅ 与 PyTorch 广播加法结果一致！")

# 也可以用省略号处理任意前导维度
y2 = einx.add("... c, c", x, bias)
assert torch.allclose(y, y2)
print("✅ 省略号写法也可以！")

# ===== 实用例子 2: LayerNorm 风格的归一化 =====
x = torch.randn(2, 10, 64)
# 对最后一维求均值，保持维度用于广播
mean = einx.mean("b s [d]", x, keepdims=True)
x_centered = x - mean
print(f"减去均值: mean shape = {mean.shape}")


添加 bias: torch.Size([4, 10, 64]) + torch.Size([64]) -> torch.Size([4, 10, 64])
✅ 与 PyTorch 广播加法结果一致！
✅ 省略号写法也可以！
减去均值: mean shape = torch.Size([2, 10, 1])


## 7. Einx 语法速查表

| 语法 | 说明 | 示例 |
|------|------|------|
| `a b c` | 命名轴 | `rearrange("a b c -> c b a", x)` |
| `(a b)` | 轴组合 | `rearrange("(a b) c -> a b c", x, a=2)` |
| `...` | 省略号 (任意维度) | `rearrange("a ... -> ... a", x)` |
| `b...` | 命名省略号 | `rearrange("(s p)... -> (s...) p...", x)` |
| `2` | 未命名轴 (指定大小) | `rearrange("a 1 b -> 1 a b 1", x)` |
| `(a + b)` | 连接 | `rearrange("h w c, h w -> h w (c + 1)", x, y)` |
| `[b]` | 操作轴 | `einx.sum("a [b] c", x)` |
| `([b])` | 保留维度的操作 | `einx.sum("a ([b]) c", x)` |

## 8. 总结

- **Einops**: 基础库，语法简单，适合入门
- **Einx**: 扩展库，完全兼容 Einops，增加了更强大的语法
- 两者可以混用，根据需求选择
- 作业文档建议：如果 Einx 有 bug，可以回退到 Einops + PyTorch
